# Credit Card Fraud Detection — Chronological validation

Dataset: public Kaggle ULB anonymized transactions. Three chronological partitions (60/20/20). Thresholds chosen on validation only. This is retrospective evaluation, not real-world validation.

In [ ]:
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
from pathlib import Path
DATA = Path("creditcard.csv")
if not DATA.exists(): DATA=Path("../creditcard.csv")
df=pd.read_csv(DATA).sort_values("Time",kind="stable").reset_index(drop=True)
print(df.shape, "Fraud:", int(df.Class.sum()), "Duplicates:", int(df.duplicated().sum()))

In [ ]:
a=int(len(df)*.6); b=int(len(df)*.8)
train,val,test=df.iloc[:a],df.iloc[a:b],df.iloc[b:]
features=[c for c in df.columns if c!="Class"]
scaler=StandardScaler()
X_train=scaler.fit_transform(train[features]); X_val=scaler.transform(val[features]); X_test=scaler.transform(test[features])
y_train=train.Class.to_numpy(); y_val=val.Class.to_numpy(); y_test=test.Class.to_numpy()
print([(len(x),int(x.Class.sum())) for x in [train,val,test]])

In [ ]:
models={"Logistic regression":LogisticRegression(class_weight="balanced",max_iter=300,random_state=42),"Hist gradient boosting":HistGradientBoostingClassifier(max_iter=110,max_depth=4,learning_rate=.08,l2_regularization=1,random_state=42)}
results=[]
for name,model in models.items():
    model.fit(X_train,y_train)
    val_scores=model.predict_proba(X_val)[:,1]
    precision,recall,thresholds=precision_recall_curve(y_val,val_scores)
    valid=np.flatnonzero(recall[:-1]>=.80)
    threshold=float(thresholds[valid[np.argmax(precision[:-1][valid])]])
    test_scores=model.predict_proba(X_test)[:,1]
    predicted=test_scores>=threshold
    tn,fp,fn,tp=confusion_matrix(y_test,predicted,labels=[0,1]).ravel()
    results.append({"model":name,"ROC-AUC":roc_auc_score(y_test,test_scores),"PR-AUC":average_precision_score(y_test,test_scores),"precision":tp/(tp+fp),"recall":tp/(tp+fn),"FPR":fp/(fp+tn),"TP":tp,"FP":fp,"FN":fn,"TN":tn,"validation_threshold":threshold})
pd.DataFrame(results)

## Interpretation and limitations
- The dataset contains anonymized PCA features and only a ~48-hour window. It cannot establish actual fraud savings or payment-system integration.
- Chronological split reduces direct future-to-past leakage; `Time` is elapsed time, not a calendar date.
- Identical transactions can exist in the data; a duplicate-aware sensitivity check is still required.
- Models have thresholds selected using validation labels only. Performance is reported on the untouched chronological test split.
- An 80% validation recall target does not guarantee 80% test recall.